In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from metrics import apk, mapk

DATA = Path("data")
con = duckdb.connect()
con.execute(f"CREATE VIEW tx AS SELECT * FROM '{(DATA / 'sample_transactions.parquet').as_posix()}'")

# Metric sanity checks
print(round(apk(["A", "B"], ["A", "X", "B"]), 3), "expected 0.833")
print(apk(["A"], ["X", "Y"]), "expected 0.0")
print(apk(["A"], ["A", "A", "A"]), "expected 1.0 (repeat guesses don't add)")
print(mapk({"c1": ["A"], "c2": ["B"]}, {"c1": ["A"]}), "expected 0.5 (c2 missing = 0)")

0.833 expected 0.833
0.0 expected 0.0
1.0 expected 1.0 (repeat guesses don't add)
0.5 expected 0.5 (c2 missing = 0)


In [2]:
labels = con.execute(f"SELECT customer_id, bought FROM '{(DATA / 'labels_valid.parquet').as_posix()}'").df()
actual = {c: list(b) for c, b in zip(labels["customer_id"], labels["bought"])}
print(f"Validation customers: {len(actual):,}")

def top_items(first_w, n_weeks, k=12):
    return con.execute(f"""
        SELECT article_id
        FROM tx
        WHERE w BETWEEN {first_w} AND {first_w + n_weeks - 1}
        GROUP BY article_id
        ORDER BY COUNT(DISTINCT customer_id) DESC, article_id
        LIMIT {k}
    """).df()["article_id"].tolist()

results = []
for n in [1, 2, 4, 8]:
    top = top_items(first_w=2, n_weeks=n)
    preds = {c: top for c in actual}
    results.append({"window_weeks": n, "MAP@12": round(mapk(actual, preds), 5)})

print(pd.DataFrame(results).to_string(index=False))

Validation customers: 14,471
 window_weeks  MAP@12
            1 0.00680
            2 0.00686
            4 0.00477
            8 0.00459


In [3]:
import numpy as np

top2 = top_items(first_w=2, n_weeks=2)
per_customer = np.array([apk(a, top2) for a in actual.values()])

rng = np.random.default_rng(42)
boot = [per_customer[rng.integers(0, len(per_customer), len(per_customer))].mean()
        for _ in range(1000)]
low, high = np.percentile(boot, [2.5, 97.5])

print(f"Popularity (2 weeks) MAP@12 = {per_customer.mean():.5f}")
print(f"95% bootstrap range:          {low:.5f} to {high:.5f}")
print(f"Customers with any hit:       {(per_customer > 0).mean():.2%}")

Popularity (2 weeks) MAP@12 = 0.00686
95% bootstrap range:          0.00605 to 0.00767
Customers with any hit:       5.35%


## Popularity baseline

The simplest thing I could think of: give every customer the same 12 best-selling items. "Best-selling" means the most different buyers, not the most rows, so a few bulk buyers can't push an item up.

I tried four time windows on the validation week. The last 2 weeks scored best (MAP@12 = 0.00686), with 1 week almost identical (0.00680). Going further back made it worse: 0.00477 for 4 weeks and 0.00459 for 8. That fits what I saw in the EDA. In fashion, last month's best-sellers are already old.

0.00686 looks tiny, but only about 5% of customers bought even one of these 12 items. That's the reality of this problem.

To see how much the score moves by chance, I resampled the validation customers 1,000 times. The 95% range was 0.00605 to 0.00767. So the 1-week vs 2-week difference is just noise. Any model I build later has to clear this range by a real margin before I call it better.

past = con.execute("""
    SELECT customer_id,
           LIST(article_id ORDER BY last_buy DESC, n DESC, article_id) AS items
    FROM (
        SELECT customer_id, article_id, MAX(t_dat) AS last_buy, COUNT(*) AS n
        FROM tx
        WHERE w >= 2
        GROUP BY customer_id, article_id
    )
    GROUP BY customer_id
""").df()
past = {c: list(i) for c, i in zip(past["customer_id"], past["items"])}

def fill(items, filler, k=12):
    out = list(items[:k])
    for f in filler:
        if len(out) >= k:
            break
        if f not in out:
            out.append(f)
    return out

preds_bb = {c: fill(past.get(c, []), top2) for c in actual}

has_hist = np.array([c in past for c in actual])
ap_bb = np.array([apk(actual[c], preds_bb[c]) for c in actual])
ap_pop = per_customer  # from Step 6.5, same customer order

print(f"{'Group':<16}{'Customers':>10}{'Popularity':>12}{'Bought-before':>15}")
for name, mask in [("All", np.ones_like(has_hist)), ("With history", has_hist), ("No history", ~has_hist)]:
    print(f"{name:<16}{mask.sum():>10,}{ap_pop[mask].mean():>12.5f}{ap_bb[mask].mean():>15.5f}")

In [4]:
past = con.execute("""
    SELECT customer_id,
           LIST(article_id ORDER BY last_buy DESC, n DESC, article_id) AS items
    FROM (
        SELECT customer_id, article_id, MAX(t_dat) AS last_buy, COUNT(*) AS n
        FROM tx
        WHERE w >= 2
        GROUP BY customer_id, article_id
    )
    GROUP BY customer_id
""").df()
past = {c: list(i) for c, i in zip(past["customer_id"], past["items"])}

def fill(items, filler, k=12):
    out = list(items[:k])
    for f in filler:
        if len(out) >= k:
            break
        if f not in out:
            out.append(f)
    return out

preds_bb = {c: fill(past.get(c, []), top2) for c in actual}

has_hist = np.array([c in past for c in actual])
ap_bb = np.array([apk(actual[c], preds_bb[c]) for c in actual])
ap_pop = per_customer  # from Step 6.5, same customer order

print(f"{'Group':<16}{'Customers':>10}{'Popularity':>12}{'Bought-before':>15}")
for name, mask in [("All", np.ones_like(has_hist)), ("With history", has_hist), ("No history", ~has_hist)]:
    print(f"{name:<16}{mask.sum():>10,}{ap_pop[mask].mean():>12.5f}{ap_bb[mask].mean():>15.5f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Group            Customers  Popularity  Bought-before
All                 14,471     0.00686        0.02447
With history        11,898     0.00688        0.02831
No history           2,573     0.00675        0.00675


In [5]:
def past_items(max_w):
    df = con.execute(f"""
        SELECT customer_id,
               LIST(article_id ORDER BY last_buy DESC, n DESC, article_id) AS items
        FROM (
            SELECT customer_id, article_id, MAX(t_dat) AS last_buy, COUNT(*) AS n
            FROM tx
            WHERE w BETWEEN 2 AND {max_w}
            GROUP BY customer_id, article_id
        )
        GROUP BY customer_id
    """).df()
    return {c: list(i) for c, i in zip(df["customer_id"], df["items"])}

rows = []
for weeks in [1, 2, 4, 8, 24]:
    p = past_items(max_w=1 + weeks)
    preds = {c: fill(p.get(c, []), top2) for c in actual}
    rows.append({"history_weeks": weeks, "MAP@12": round(mapk(actual, preds), 5)})

print(pd.DataFrame(rows).to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 history_weeks  MAP@12
             1 0.02064
             2 0.02300
             4 0.02411
             8 0.02463
            24 0.02447


In [6]:
partners_df = con.execute("""
    WITH b AS (
        SELECT DISTINCT customer_id, t_dat, article_id
        FROM tx WHERE w BETWEEN 2 AND 9
    ),
    p AS (
        SELECT x.article_id AS item, y.article_id AS partner, COUNT(*) AS n
        FROM b x
        JOIN b y ON x.customer_id = y.customer_id
                AND x.t_dat = y.t_dat
                AND x.article_id <> y.article_id
        GROUP BY ALL
    ),
    r AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY item ORDER BY n DESC, partner) AS rk
        FROM p
    )
    SELECT item, LIST(partner ORDER BY rk) AS partners
    FROM r WHERE rk <= 3
    GROUP BY item
""").df()
partners = {i: list(p) for i, p in zip(partners_df["item"], partners_df["partners"])}
print(f"Items with partners: {len(partners):,}")

p8 = past_items(max_w=9)

def with_partners(items):
    out = list(items)
    for it in items:
        for q in partners.get(it, []):
            if q not in out:
                out.append(q)
    return out

preds_b8 = {c: fill(p8.get(c, []), top2) for c in actual}
preds_bt = {c: fill(with_partners(p8.get(c, [])), top2) for c in actual}

ap_b8 = np.array([apk(actual[c], preds_b8[c]) for c in actual])
ap_bt = np.array([apk(actual[c], preds_bt[c]) for c in actual])

print(f"{'Group':<16}{'Bought-before 8w':>18}{'+ Bought-together':>19}")
for name, mask in [("All", np.ones_like(has_hist)), ("With history", has_hist), ("No history", ~has_hist)]:
    print(f"{name:<16}{ap_b8[mask].mean():>18.5f}{ap_bt[mask].mean():>19.5f}")

Items with partners: 24,852
Group             Bought-before 8w  + Bought-together
All                        0.02463            0.02554
With history               0.02850            0.02960
No history                 0.00675            0.00675


In [7]:
def paired_gain(ap_new, ap_old, n_boot=1000, seed=42):
    diff = ap_new - ap_old
    rng = np.random.default_rng(seed)
    boot = [diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(n_boot)]
    low, high = np.percentile(boot, [2.5, 97.5])
    return diff.mean(), low, high, (diff > 0).sum(), (diff < 0).sum()

gain, low, high, better, worse = paired_gain(ap_bt, ap_b8)
print(f"Mean gain:       {gain:+.5f}")
print(f"95% range:       {low:+.5f} to {high:+.5f}")
print(f"Customers better: {better:,}   worse: {worse:,}")
print("Real gain" if low > 0 else "Not clearly real (range includes 0)")

Mean gain:       +0.00091
95% range:       +0.00059 to +0.00127
Customers better: 248   worse: 236
Real gain


## "Bought before" and "bought together"

For "bought before", I recommend a customer's own past purchases, most recent first, and fill any empty slots with the 2-week best-sellers. I expected this to be weak, because the EDA showed only 3.9% of purchases are exact repeats. I was wrong. It scored 0.02447, about 3.6 times the popularity baseline. For customers with history it reached 0.02831. Customers without history get plain popularity, so their score didn't change, which is a useful sanity check.

I think the reason is that MAP@12 rewards getting a few items right near the top. Popularity almost never does that. A customer's own recent purchases sometimes do.

How far back to look mattered a bit. One week of history was clearly worse (0.02064). Anything from 4 to 24 weeks was close, with 8 weeks the best at 0.02463. History older than that added nothing, so cutting the data to 26 weeks in Step 4 probably didn't cost this method much.

For "bought together", I found which items people buy on the same day and added each past item's top 3 partners after the past items. This moved MAP@12 from 0.02463 to 0.02554. To check it wasn't luck, I bootstrapped the difference on the same customers. The 95% range was +0.00059 to +0.00127, so the gain is real. But it's small and it only changed the result for about 3% of customers, and almost as many got worse as got better.

Best classic method so far: past items (8 weeks), then bought-together partners, then best-sellers. MAP@12 = 0.02554 on validation.